In [ ]:
"""

Arquitetura base - GPT-2

"""

import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

    
class LayerNorm(nn.Module):
    """LayerNorm com suporte a bias opcional."""
    
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.ndim = ndim  # Guarda a dimensão
        #self.eps = eps  # Guarda o valor de epsilon
    def forward(self, input):
        # CORREÇÃO: usar a dimensão correta (a última dimensão)
        return F.layer_norm(input, (self.ndim,), self.weight, self.bias, 1e-5)

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = MultiHead(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MultiHead(nn.Module):
    """Implementação MultiHead Attention para teste."""
    
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.num_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.num_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, 
                dropout_p=self.dropout if self.training else 0, 
                is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
            
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class ModeloCompleto(nn.Module):
    """Versão simplificada do modelo para teste."""
    
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        # Embeddings
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        
        # Blocks
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.num_layer)])
        
        # Final layer norm
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)
        
        # Language model head
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Weight tying
        self.wte.weight = self.lm_head.weight
        
        # Inicialização
        self.apply(self._init_weights)
        
        # Inicialização especial para projeções residuais
        for name, param in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(param, mean=0.0, std=0.02/math.sqrt(2 * config.num_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.config.block_size
        
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        
        return logits, loss


    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx





# Configuração do modelo
class Config:
    def __init__(self, config_dict):
        self.vocab_size = config_dict.get("vocab_size", 10000)
        self.n_embd = config_dict.get("embedding_dim", 128)
        self.num_head = config_dict.get("num_heads", 2)
        self.num_layer = config_dict.get("num_layers", 2)
        self.dropout = config_dict.get("dropout", 0.0)
        self.bias = config_dict.get("bias", False)
        self.block_size = config_dict.get("block_size", config_dict.get("seq_len", 128))
        self.num_experts = config_dict.get("num_experts", 8)
        self.num_experts_per_tok = config_dict.get("num_experts_per_tok", 4)
        self.moe_aux_loss_coef = config_dict.get("moe_aux_loss_coef", 0.01)

# Save final
def save_model(model_save_path, model):
    """Salva o modelo em FP16"""
    Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)
    state_dict = model.state_dict()
    fp16_state_dict = {}
    for key, value in state_dict.items():
        if value.is_floating_point():
            fp16_state_dict[key] = value.half()
        else:
            fp16_state_dict[key] = value
    
    torch.save(fp16_state_dict, model_save_path)
    print(f"Modelo salvo em FP16: {model_save_path}")

# Learning rate dinamico - boa pratica
def get_lr( step: int, lr: float, warmup_steps: int, max_steps: int, min_lr: float) -> float:
    """Warmup linear seguido de cosine decay até min_lr."""
    if warmup_steps > 0 and step < warmup_steps:
        return min(lr * (step + 1) / warmup_steps, lr)

    if step >= max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    # Protege contra um erro de arredondamento de ponto flutuante na fronteira.
    return min(max(min_lr + coeff * (lr - min_lr), min_lr), lr)

@torch.no_grad()
def estimate_loss(model, loader, num_batches: int, device, amp_dtype, use_amp: bool) -> float:
    """Calcula a loss media em batches reservados para validacao."""
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = loader.get_batch()
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            _, loss = model(x, y)
        losses.append(loss.item())
    if was_training:
        model.train()
    return sum(losses) / len(losses)


# Save final
def save_model(model_save_path, model):
    """Salva o modelo em FP16"""
    path = Path(model_save_path)
    if str(path.parent) != ".":
        path.parent.mkdir(parents=True, exist_ok=True)
    
    state_dict = model.state_dict()
    fp16_state_dict = {}
    for key, value in state_dict.items():
        if value.is_floating_point():
            fp16_state_dict[key] = value.half()
        else:
            fp16_state_dict[key] = value
    
    torch.save(fp16_state_dict, path)
    print(f"Modelo salvo em FP16: {path}")


In [ ]:
# #!/usr/bin/env python3
# """
# Versão minimalista para baixar train.bin
# """

# import os
# from huggingface_hub import hf_hub_download

# # Configure aqui
# REPO_ID = "marcos-j-leemes/tinyS"
# TOKEN = "SEU_TOKEN_AQUI"  # Coloque seu token aqui | acelera o downloading
# FILENAME = "train.bin"

# # Baixa
# print(f"Baixando {FILENAME}...")
# path = hf_hub_download(
#     repo_id=REPO_ID,
#     filename=FILENAME,
#     repo_type="dataset",
#     token=TOKEN,
#     local_dir=".",
# )

# print(f" Baixado para: {path}")

Baixando train.bin...


train.bin:   0%|          | 0.00/920M [00:00<?, ?B/s]

 Baixado para: train.bin


In [ ]:
"""
Treinamento com múltiplas GPUs usando FullyShardedDataParallel (FSDP) — equivalente
ao ZeRO (DeepSpeed) nativo do PyTorch.

Rodar com:
    torchrun --nproc_per_node=2 train_fsdp.py

Principais mudanças em relação à versão DDP:
  1. FSDP no lugar de DDP: em vez de cada GPU guardar uma cópia COMPLETA do
     modelo + gradientes + estado do otimizador, cada GPU guarda só a SUA
     fatia (shard). O restante é buscado via all-gather sob demanda no
     forward/backward e descartado logo em seguida.
  2. sharding_strategy define o "nível de ZeRO":
       - ShardingStrategy.FULL_SHARD    -> equivalente a ZeRO-3 (shard de
         params, grads e optimizer states). Máxima economia de memória.
       - ShardingStrategy.SHARD_GRAD_OP -> equivalente a ZeRO-2 (shard de
         grads e optimizer states, params replicados). Menos comunicação,
         mais memória.
       - ShardingStrategy.NO_SHARD      -> equivalente ao DDP puro.
       - ShardingStrategy.HYBRID_SHARD  -> shard dentro do nó, replica entre
         nós (bom para multi-node).
  3. auto_wrap_policy: decide COMO o modelo é particionado em unidades FSDP.
     Sem isso, o modelo inteiro vira uma única unidade e você perde boa
     parte do benefício de memória. O ideal é envolver cada bloco do
     transformer individualmente — ajuste `transformer_auto_wrap_policy`
     abaixo para apontar para a classe real do seu bloco (ex: Block,
     TransformerBlock, DecoderLayer...). Deixei um fallback por tamanho
     (size_based_auto_wrap_policy) caso você não tenha essa classe à mão.
  4. Clipping de gradiente: NÃO use mais
     torch.nn.utils.clip_grad_norm_(model.parameters(), ...) — com params
     fatiados isso dá norma errada. Use o método próprio do FSDP:
     model.clip_grad_norm_(GRAD_CLIP).
  5. model.module não existe mais como "unwrap direto". Para obter o
     state_dict completo (ex: para salvar), é preciso usar o context
     manager FSDP.state_dict_type(...) com FullStateDictConfig, que faz um
     all-gather e materializa o state dict completo (por padrão só no
     rank 0, com offload pra CPU).
  6. mixed_precision: se quiser usar BF16/FP16, isso é configurado via
     MixedPrecision no próprio FSDP (em vez de autocast manual), pois o
     FSDP também precisa saber em que dtype fazer o all-gather.
  7. device_id=local_rank: substitui o .to(device) + device_ids do DDP —
     o FSDP já move os shards para a GPU certa internamente.
"""

# imports library
import functools
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
    BackwardPrefetch,
    FullStateDictConfig,
    StateDictType,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import os
import time
import numpy as np

# Dados
DATASET = ""# os.path.dirname(os.path.abspath(__file__))
FILENAME = "train.bin"

# Configurações do modelo
VOCAB_SIZE = 10001
EMBEDDING_DIM = 128
NUM_HEADS = 2
NUM_LAYERS = 4
BLOCK_SIZE = 260
DROPOUT = 0.0

# configurações do treinamento
MAX_STEPS = 1000
BATCH_SIZE = 12          # batch LOCAL por GPU
GRAD_ACCUM_STEPS = 20
WEIGHT_DECAY = 0.0001
WARMUP_STEPS = 10
LEARNING_RATE = 1e-3
MIN_LR = 1e-5
GRAD_CLIP = 1.0

# logs
PRINT_INTERVAL = 10
EVAL_INTERVAL = 100

# Configuração do hardware
USE_AMP = False
USE_BF16 = False           # se True, ativa MixedPrecision do FSDP em bf16
TORCH_COMPILE = False       # se for usar torch.compile, use_orig_params=True abaixo é obrigatório

# ZeRO stage / sharding strategy
# "FULL_SHARD" = ZeRO-3 | "SHARD_GRAD_OP" = ZeRO-2 | "NO_SHARD" = DDP
SHARDING_STRATEGY = ShardingStrategy.FULL_SHARD

# save model
STEP_SAVE_INTERVAL = 500
SAVE_DIR = "model_final.pth"


# ---------------------------------------------------------------------------
# setup do process group + leitura de rank/local_rank/world_size (igual DDP)
# ---------------------------------------------------------------------------
def ddp_setup():
    dist.init_process_group(backend="nccl")
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    return rank, local_rank, world_size


def ddp_cleanup():
    dist.destroy_process_group()


# ---------------------------------------------------------------------------
# get_batch idêntico ao da versão DDP: cada rank sorteia só na sua fatia
# lógica do dataset.
# ---------------------------------------------------------------------------
data_dir = DATASET
def get_batch(split, batch_size, block_size, device, rank, world_size):
    filename = FILENAME if split == 'train' else 'val.bin'
    path = os.path.join(data_dir, filename)
    if split == 'val' and not os.path.exists(path):
        path = os.path.join(data_dir, FILENAME)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo de dados nao encontrado: {path}")

    data = np.memmap(path, dtype=np.uint16, mode='r')
    total_len = len(data) - block_size
    if total_len <= 0:
        raise ValueError(f"{path} possui poucos tokens para BLOCK_SIZE={block_size}.")

    shard_len = total_len // world_size
    shard_start = rank * shard_len
    shard_end = shard_start + shard_len

    ix = torch.randint(shard_start, shard_end, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y


@torch.no_grad()
def evaluate(model, device, rank, world_size, num_batches=10):
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch("val", BATCH_SIZE, BLOCK_SIZE, device, rank, world_size)
        _, loss = model(x, y)
        losses.append(loss.item())
    if was_training:
        model.train()
    return sum(losses) / len(losses)


# ---------------------------------------------------------------------------
# Salvar checkpoint com FSDP: precisa "des-fatiar" (all-gather) os params
# antes de chamar state_dict(). offload_to_cpu + rank0_only evitam estourar
# a memória da GPU/host durante esse gather.
# ---------------------------------------------------------------------------
def save_fsdp_model(path, model, is_main_process):
    save_policy = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_policy):
        full_state_dict = model.state_dict()

    if is_main_process:
        # Ajuste aqui conforme a assinatura real do seu save_model():
        # se ele espera um objeto nn.Module (com .state_dict()), construa
        # uma instância "crua" do modelo e dê load_state_dict antes de salvar.
        torch.save(full_state_dict, path)
        print(f"Checkpoint salvo em: {path}")


def main():
    rank, local_rank, world_size = ddp_setup()
    device = torch.device(f"cuda:{local_rank}")
    is_main_process = (rank == 0)

    config = Config({
        "vocab_size": VOCAB_SIZE,
        "embedding_dim": EMBEDDING_DIM,
        "num_heads": NUM_HEADS,
        "num_layers": NUM_LAYERS,
        "block_size": BLOCK_SIZE,
        "dropout": DROPOUT,
    })

    # IMPORTANTE: com FSDP, NÃO faça model.to(device) antes de envolver com
    # FSDP quando usar device_id — o FSDP cuida do posicionamento na GPU
    # correta ao materializar cada shard. (Se seu modelo for grande demais
    # para caber inteiro numa GPU só ao ser instanciado, considere criar em
    # meta device — fora do escopo deste ajuste pontual.)
    model = ModeloCompleto(config)

    # -----------------------------------------------------------------
    # auto_wrap_policy: troque pelo bloco real do seu transformer se
    # tiver a classe disponível, ex:
    #
    #   from main import Block
    #   from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
    #   auto_wrap_policy = functools.partial(
    #       transformer_auto_wrap_policy,
    #       transformer_layer_cls={Block},
    #   )
    #
    # Isso garante que cada bloco vira sua própria unidade FSDP (melhor
    # granularidade = melhor overlap de comunicação/computação e menos
    # pico de memória). O fallback abaixo funciona sem conhecer a classe,
    # mas é menos eficiente.
    # -----------------------------------------------------------------
    auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy, min_num_params=1_000_000
    )

    mixed_precision_policy = None
    if USE_BF16:
        mixed_precision_policy = MixedPrecision(
            param_dtype=torch.bfloat16,
            reduce_dtype=torch.bfloat16,
            buffer_dtype=torch.bfloat16,
        )

    model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        sharding_strategy=SHARDING_STRATEGY,
        mixed_precision=mixed_precision_policy,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=local_rank,
        use_orig_params=TORCH_COMPILE,  # obrigatório se for usar torch.compile
    )

    if is_main_process:
        parametros = int(sum(p.numel() for p in model.parameters()))
        print(f"Modelo: FSDP ({SHARDING_STRATEGY.name}) | Parâmetros: {parametros:,}")
        print(f"World size: {world_size} | Batch local: {BATCH_SIZE} | "
              f"Batch efetivo global: {BATCH_SIZE * world_size}")
        print(f"{MAX_STEPS} steps | Grad Accum Steps: {GRAD_ACCUM_STEPS} | "
              f"LR: {LEARNING_RATE} | Min LR: {MIN_LR} | Warmup Steps: {WARMUP_STEPS}")
        print()

    # O otimizador SEMPRE deve ser criado DEPOIS de envolver o modelo com
    # FSDP, pois model.parameters() aqui já reflete os params fatiados.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=float(WEIGHT_DECAY),
    )
    
    model.train()
    step = 0
    val_loss = 0.0
    tokens_seen = 0
    
    tokens_per_step = BLOCK_SIZE * BATCH_SIZE * GRAD_ACCUM_STEPS * world_size
    
    while step < MAX_STEPS:
        torch.cuda.synchronize()
        t0 = time.time()
    
        optimizer.zero_grad(set_to_none=True)
        step_loss_accum = 0.0
    
        lr = get_lr(step, LEARNING_RATE, WARMUP_STEPS, MAX_STEPS, MIN_LR)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
    
        for micro_step in range(GRAD_ACCUM_STEPS):
            x, y = get_batch("train", BATCH_SIZE, BLOCK_SIZE, device, rank, world_size)
    
            logits, loss = model(x, y)
    
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss não finita no step {step}: {loss.item()}")
    
            step_loss_accum += loss.item()
    
            loss_scaled = loss / GRAD_ACCUM_STEPS
            loss_scaled.backward()
            # Reduce-scatter dos gradientes acontece aqui dentro do .backward()
    
        grad_norm = 0.0
        if GRAD_CLIP is not None:
            # Clipping via método próprio do FSDP (obrigatório com params fatiados)
            grad_norm = model.clip_grad_norm_(GRAD_CLIP)
    
        optimizer.step()
    
        torch.cuda.synchronize()
        dt = time.time() - t0
        tokens_seen += tokens_per_step
        tokens_per_sec = tokens_per_step / dt
    
        # -----------------------------------------------------------------
        # CORREÇÃO: evaluate() faz forward -> comunicação coletiva (all-gather
        # de parâmetros com FULL_SHARD). TODOS os ranks precisam chamar,
        # senão o rank 0 fica esperando os outros pra sempre (deadlock).
        # Só o PRINT fica restrito ao rank principal.
        # -----------------------------------------------------------------
        if step % EVAL_INTERVAL == 0:
            val_loss = evaluate(model, device, rank, world_size)  # todos os ranks
            if is_main_process:
                print(f"Validação | Step {step} | Val Loss: {val_loss:.4f} | LR: {lr:.10f}")
    
        if is_main_process and step % PRINT_INTERVAL == 0:
            print(
                f"Step {step} | Loss: {step_loss_accum / GRAD_ACCUM_STEPS:.4f} | "
                f"VAL Loss: {val_loss:.4f} | LR: {lr:.10f} | "
                f"norm: {grad_norm:.4f} | dt: {dt*1000:.2f}ms | "
                f"tok/s: {tokens_per_sec:,.0f} | tokens vistos: {tokens_seen:,}"
            )
    
        step += 1
    
    if is_main_process:
        print("Treinamento finalizado.")
    
    # save_fsdp_model já lida com o all-gather + salvar só no rank 0
    save_fsdp_model(SAVE_DIR, model, is_main_process)
    
    ddp_cleanup()


if __name__ == "__main__":
    main()
    exit()

In [18]:
# Full_SHARD
!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb
# torchrun --nproc_per_node=2 train_ddp.py

W0908 16:56:54.955000 777 torch/distributed/run.py:852] 
W0908 16:56:54.955000 777 torch/distributed/run.py:852] *****************************************
W0908 16:56:54.955000 777 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0908 16:56:54.955000 777 torch/distributed/run.py:852] *****************************************
Modelo: FSDP (FULL_SHARD) | Parâmetros: 1,690,560
World size: 2 | Batch local: 12 | Batch efetivo global: 24
1000 steps | Grad Accum Steps: 20 | LR: 0.001 | Min LR: 1e-05 | Warmup Steps: 10

Validação | Step 0 | Val Loss: 9.1395 | LR: 0.0001000000
Step 0 | Loss: 9.2218 | VAL Loss: 9.1395 | LR: 0.0001000000 | norm: 1.5327 | dt: 1281.42ms | tok/s: 97,392 | tokens vistos: 124,800
Step 10 | Loss: 8.0819 | VAL Loss: 9.1395 | LR: 0.0010000000 | norm: 1.3673 | dt: 466.01ms | tok/s: 2

In [16]:
# Full_SHARD
!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb
# torchrun --nproc_per_node=2 train_ddp.py

W0908 16:47:59.547000 668 torch/distributed/run.py:852] 
W0908 16:47:59.547000 668 torch/distributed/run.py:852] *****************************************
W0908 16:47:59.547000 668 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0908 16:47:59.547000 668 torch/distributed/run.py:852] *****************************************
Modelo: FSDP (FULL_SHARD) | Parâmetros: 1,500,096
World size: 2 | Batch local: 12 | Batch efetivo global: 24
500 steps | Grad Accum Steps: 20 | LR: 0.001 | Min LR: 1e-05 | Warmup Steps: 10

Validação | Step 0 | Val Loss: 9.1930 | LR: 0.0001000000
Step 0 | Loss: 9.2395 | VAL Loss: 9.1930 | LR: 0.0001000000 | norm: 1.0704 | dt: 1314.88ms | tok/s: 131,419 | tokens vistos: 172,800
Step 10 | Loss: 8.1507 | VAL Loss: 9.1930 | LR: 0.0010000000 | norm: 1.3688 | dt: 466.16ms | tok/s: 3

In [10]:
# SHARD_GRAD_OP
!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb
# torchrun --nproc_per_node=2 train_ddp.py

W0908 16:36:51.708000 386 torch/distributed/run.py:852] 
W0908 16:36:51.708000 386 torch/distributed/run.py:852] *****************************************
W0908 16:36:51.708000 386 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0908 16:36:51.708000 386 torch/distributed/run.py:852] *****************************************
Modelo: FSDP (FULL_SHARD) | Parâmetros: 1,483,456
World size: 2 | Batch local: 12 | Batch efetivo global: 24
500 steps | Grad Accum Steps: 1 | LR: 0.001 | Min LR: 1e-05 | Warmup Steps: 10

^C
W0908 16:40:24.590000 386 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0908 16:40:24.591000 386 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 392 closing signal SIGINT
W0908 16:40:24.592000 386 torch/distribute

In [12]:
# NO_SHARD
!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb
# torchrun --nproc_per_node=2 train_ddp.py

W0908 16:41:10.759000 467 torch/distributed/run.py:852] 
W0908 16:41:10.759000 467 torch/distributed/run.py:852] *****************************************
W0908 16:41:10.759000 467 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0908 16:41:10.759000 467 torch/distributed/run.py:852] *****************************************
/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/wrap.py:485: FutureWarning: The `NO_SHARD` sharding strategy is deprecated. If having issues, please use `DistributedDataParallel` instead.
  return wrapper_cls(module, **kwargs)
/kaggle/working/.virtual_documents/__notebook_source__.ipynb:533: FutureWarning: The `NO_SHARD` sharding strategy is deprecated. If having issues, please use `DistributedDataParallel` instead.
  model = FSDP(
Modelo: FSDP (NO_SHARD) | Parâ